# SQL and pandas, side by side

Four questions, each answered both ways, with the two results compared so you
can see they agree.

The verbs are the same in both: **filter, group, aggregate, join, sort**.
Every tool in this area has its own words for those five.

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")
plays = pd.read_sql("SELECT * FROM plays", con, parse_dates=["played_at"])
artists = pd.read_sql("SELECT * FROM artists", con)
pd.set_option("display.width", 110)

## 1. Plays per device

```sql
SELECT device, COUNT(*) AS plays
FROM plays
GROUP BY device
ORDER BY plays DESC;
```

In [ ]:
sql = pd.read_sql("""
    SELECT device, COUNT(*) AS plays
    FROM plays GROUP BY device ORDER BY plays DESC
""", con)

pandas_way = (plays["device"].value_counts()
                             .rename_axis("device")
                             .reset_index(name="plays"))

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))
print("\nsame answer:", sql.equals(pandas_way))

If one of those comparisons prints `False` on your machine, the numbers are almost
certainly fine and the **types** differ. `.equals()` is strict about that. Print
`sql.dtypes` and `pandas_way.dtypes` and compare: older pandas versions return text
columns as `object` where newer ones return `str`, which is enough to make `.equals()`
say no. Comparing `.to_string()` output, or `.values.tolist()`, is more forgiving.

## 2. Plays over 8 minutes

```sql
SELECT COUNT(*) FROM plays WHERE minutes_played > 8;
```

In [ ]:
sql_count = pd.read_sql(
    "SELECT COUNT(*) AS n FROM plays WHERE minutes_played > 8", con
)["n"][0]

pandas_count = len(plays[plays["minutes_played"] > 8])

print(sql_count, pandas_count, "| same answer:", sql_count == pandas_count)

## 3. Average minutes per genre

Genre lives in the `artists` table, so both versions need both tables.

```sql
SELECT a.genre, ROUND(AVG(p.minutes_played), 2) AS avg_min
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY a.genre
ORDER BY avg_min DESC;
```

In [ ]:
sql = pd.read_sql("""
    SELECT a.genre, ROUND(AVG(p.minutes_played), 2) AS avg_min
    FROM plays p
    JOIN artists a USING (artist_id)
    GROUP BY a.genre
    ORDER BY avg_min DESC
""", con)

pandas_way = (plays.merge(artists, on="artist_id")
                   .groupby("genre")["minutes_played"]
                   .mean()
                   .round(2)
                   .sort_values(ascending=False)
                   .reset_index(name="avg_min"))

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))
print("\nsame answer:", sql.equals(pandas_way))

## 4. Top five artists by minutes

```sql
SELECT a.artist_name, ROUND(SUM(p.minutes_played), 1) AS minutes
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY a.artist_id
ORDER BY minutes DESC
LIMIT 5;
```

In [ ]:
sql = pd.read_sql("""
    SELECT a.artist_name, ROUND(SUM(p.minutes_played), 1) AS minutes
    FROM plays p
    JOIN artists a USING (artist_id)
    GROUP BY a.artist_id
    ORDER BY minutes DESC
    LIMIT 5
""", con)

pandas_way = (plays.merge(artists, on="artist_id")
                   .groupby("artist_name")["minutes_played"]
                   .sum()
                   .round(1)
                   .nlargest(5)
                   .reset_index(name="minutes"))

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))
print("\nsame answer:", sql.equals(pandas_way))

## A translation table

| Idea | SQL | pandas |
|---|---|---|
| pick columns | `SELECT a, b` | `df[["a", "b"]]` |
| pick rows | `WHERE x > 1` | `df[df["x"] > 1]` |
| two conditions | `AND` / `OR` | `&` / `|`, with brackets |
| one of a set | `IN (...)` | `.isin([...])` |
| missing values | `IS NULL` | `.isna()` |
| sort | `ORDER BY x DESC` | `.sort_values("x", ascending=False)` |
| first n | `LIMIT 5` | `.head(5)` or `.nlargest(5)` |
| distinct values | `SELECT DISTINCT x` | `df["x"].unique()` |
| count per value | `GROUP BY x` + `COUNT(*)` | `df["x"].value_counts()` |
| group and aggregate | `GROUP BY x` | `.groupby("x").agg(...)` |
| filter groups | `HAVING` | filter the result after `.groupby()` |
| join | `JOIN ... ON` | `.merge(other, on=...)` |
| left join | `LEFT JOIN` | `.merge(..., how="left")` |
| rename output | `AS name` | `.rename()`, or named `agg` |

## So which should you use?

**Reach for SQL when** the data already lives in a database, you want an
answer rather than a program, the data is bigger than your memory, or
somebody else needs to run the same query.

**Reach for pandas when** you need to clean or reshape rather than select,
you want a chart at the end, the logic is awkward in SQL, or it is part of a
larger Python program.

**In practice, both, in that order.** `pd.read_sql` exists precisely for
this: let the database do the filtering and joining, where it is fast and
where the data lives, then hand pandas a smaller tidier table to clean,
reshape and chart. That is what session 10's pipeline does.

People get strangely tribal about this. It is a choice of tool, not of team.

In [ ]:
con.close()